# DeepSpeed Configuration Tutorial

## Overview

DeepSpeed is Microsoft's deep learning optimization library for large-scale distributed training.

### Learning Objectives
- Configure ZeRO stages for memory optimization
- Set up mixed precision training
- Enable CPU/NVMe offloading

### References
- [DeepSpeed Documentation](https://www.deepspeed.ai/)
- Rajbhandari et al., "ZeRO: Memory Optimizations Toward Training Trillion Parameter Models", SC 2020

## 1. DeepSpeed Configuration Structure

```json
{
  "train_batch_size": 32,
  "gradient_accumulation_steps": 1,
  "fp16": { "enabled": true },
  "zero_optimization": { "stage": 2 },
  "optimizer": { "type": "AdamW" }
}
```

In [ ]:
import json

def create_deepspeed_config(zero_stage=2, fp16=True, offload=False):
    """Create DeepSpeed configuration."""
    config = {
        "train_batch_size": "auto",
        "train_micro_batch_size_per_gpu": "auto",
        "gradient_accumulation_steps": "auto",
        "fp16": {"enabled": fp16, "loss_scale": 0, "initial_scale_power": 16},
        "zero_optimization": {
            "stage": zero_stage,
            "overlap_comm": True,
            "contiguous_gradients": True,
            "reduce_bucket_size": 5e7,
        },
        "optimizer": {
            "type": "AdamW",
            "params": {"lr": 1e-4, "betas": [0.9, 0.999], "weight_decay": 0.01}
        }
    }
    
    if offload:
        config["zero_optimization"]["offload_optimizer"] = {"device": "cpu"}
        if zero_stage == 3:
            config["zero_optimization"]["offload_param"] = {"device": "cpu"}
    
    return config

print("ZeRO Stage 2 Config:")
print(json.dumps(create_deepspeed_config(2), indent=2))

## 2. Training with DeepSpeed

```python
import deepspeed

model_engine, optimizer, _, _ = deepspeed.initialize(
    model=model,
    config="ds_config.json"
)

for batch in dataloader:
    loss = model_engine(batch)
    model_engine.backward(loss)
    model_engine.step()
```

## 3. Summary

| ZeRO Stage | Memory Savings | Use Case |
|------------|----------------|----------|
| Stage 1 | ~4x | Models up to 10B |
| Stage 2 | ~8x | Models 10-50B |
| Stage 3 | Linear | Models 50B+ |
| + Offload | Additional | Memory constrained |